In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_core.documents import Document

QDRANT_URL = ""
GEMINI_API_KEY = ""

In [ ]:
df = pd.read_parquet("./tmdb_movies_2021_2025.parquet")

df['genres_list'] = df['genres'].apply(lambda x: str(x).split('|') if pd.notna(x) else [])


df_clean = df[
    (df['vote_count'] > 100) & 
    (df['overview'].notna()) & 
    (df['title'].notna())
].copy()


df_demo = df_clean.sort_values('popularity', ascending=False).head(300)

print(f"Total Movies: {len(df_demo)}")

In [ ]:
client = QdrantClient(url=QDRANT_URL)
collection_name = "movies_hybrid"

dense_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001", 
    google_api_key=GEMINI_API_KEY
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    }
)

print(f"Collection '{collection_name}' created with Hybrid support (Dense + Sparse).")

In [ ]:
import time

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    sparse_vector_name="sparse"
)

docs = []
for _, row in df_demo.iterrows():

    metadata = {
        "title": row["title"],
        "release_date": row["release_date"],
        "genres": row["genres_list"],
        "vote_average": row["vote_average"],
        "poster_url": row["poster_url"]
    }
    

    content = f"Title: {row['title']}, \nOverview: {row['overview']}"
    
    docs.append(Document(page_content=content, metadata=metadata))

In [ ]:
batch_size = 80
total_batches = (len(docs) // batch_size) + 1

print(f"Initializing {total_batches} batches")

for i in range(0, len(docs), batch_size):
    batch = docs[i : i + batch_size]
    batch_n = (i // batch_size) + 1
    
    print(f"Indexing {batch_n}/{total_batches} ({len(batch)} movies)...")
    vector_store.add_documents(batch)
    

    if i + batch_size < len(docs):
        print("Limit reached, waiting 60 seconds to refresh Gemini free quota...")
        time.sleep(60)

print(f"Done, check {QDRANT_URL}/dashboard")

Iniciando inyección en 4 lotes para respetar los límites de la API...
🚀 Inyectando lote 1/4 (80 películas)...
⏳ Límite alcanzado. Esperando 60 segundos para refrescar la cuota gratuita de Gemini...
🚀 Inyectando lote 2/4 (80 películas)...
⏳ Límite alcanzado. Esperando 60 segundos para refrescar la cuota gratuita de Gemini...
🚀 Inyectando lote 3/4 (80 películas)...
⏳ Límite alcanzado. Esperando 60 segundos para refrescar la cuota gratuita de Gemini...
🚀 Inyectando lote 4/4 (60 películas)...
✅ ¡Ingesta híbrida completada con éxito! Revisa tu dashboard en http://localhost:6333/dashboard
